In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.metrics import classification_report, accuracy_score

from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier, 
    StackingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

df.to_csv("../data/processed/cleaned_floods.csv")
df = pd.DataFrame(data)
df.head()

,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,Encroachments,DrainageSystems,Landslides,Watersheds,DeterioratingInfrastructure,WetlandLoss,InadequatePlanning,FloodProbability
0,3,8.0,6.0,6,4,4,6.0,2,2.0,10.0,4,2.0,3.0,3.0,2.0,0.450
1,8,4.0,5.0,7,7,9,1.0,5,4.0,9.0,6,2.0,1.0,9.0,1.0,0.475
2,3,10.0,4.0,1,7,5,4.0,7,9.0,7.0,4,8.0,6.0,8.0,3.0,0.515
3,4,4.0,2.0,7,3,4,1.0,4,4.0,4.0,6,6.0,8.0,6.0,6.0,0.520
4,3,7.0,5.0,2,5,8,5.0,2,5.0,7.0,5,3.0,3.0,4.0,3.0,0.475


In [3]:
Features = ['MonsoonIntensity', 'TopographyDrainage', 'RiverManagement',
       'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality',
       'Siltation', 'Encroachments', 'DrainageSystems', 'Landslides',
       'Watersheds', 'DeterioratingInfrastructure', 'WetlandLoss',
       'InadequatePlanning']
# Feature engineering
df['InfraScore'] = df['DamsQuality'] + df['DrainageSystems'] + df['DeterioratingInfrastructure'] + df['RiverManagement']
df['EnvRisk'] = df['Deforestation'] + df['WetlandLoss'] + df['Urbanization'] + df['Encroachments']
df['ClimateRisk'] = df['MonsoonIntensity'] + df['ClimateChange'] + df['Landslides'] + df['Siltation']
df['RiskRatio'] = df['EnvRisk'] / (df['InfraScore'] + 1)
df['TotalScore'] = df[Features].sum(axis=1)
Features = Features + ['InfraScore', 'EnvRisk', 'ClimateRisk', 'RiskRatio', 'TotalScore']
X = df[Features]
median = df['FloodProbability'].median()
y = (df['FloodProbability'] >= median).astype(int)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [5]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier

# Base models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=200, random_state=42)),
]

# Meta model
meta_model = RandomForestClassifier(n_estimators=200, random_state=42)

# Stack
stacked_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

# Train
stacked_model.fit(X_train, y_train)

# Predict
y_pred = stacked_model.predict(X_test)

# Results
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.7838
              precision    recall  f1-score   support

           0       0.78      0.78      0.78      4821
           1       0.79      0.79      0.79      5179

    accuracy                           0.78     10000
   macro avg       0.78      0.78      0.78     10000
weighted avg       0.78      0.78      0.78     10000



In [6]:
risk = {0: "Low", 1: "High"}


In [7]:
test_data = np.array([
    [3, 5, 6, 2, 4, 3, 7, 4, 2, 6, 1, 5, 6, 3, 4],
    [6, 6, 5, 5, 6, 6, 5, 6, 5, 5, 4, 6, 5, 5, 6],
    [9, 3, 3, 8, 8, 8, 3, 9, 7, 2, 7, 4, 3, 8, 7],
    [10, 2, 2, 9, 9, 9, 2, 10, 9, 1, 8, 3, 2, 9, 9],
    [8, 8, 8, 3, 4, 7, 9, 5, 3, 9, 2, 8, 8, 3, 4]
])

# Original 15 feature names
base_features = ['MonsoonIntensity', 'TopographyDrainage', 'RiverManagement',
       'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality',
       'Siltation', 'Encroachments', 'DrainageSystems', 'Landslides',
       'Watersheds', 'DeterioratingInfrastructure', 'WetlandLoss',
       'InadequatePlanning']

test_df = pd.DataFrame(test_data, columns=base_features)

# Compute the same engineered features
test_df['InfraScore'] = test_df['DamsQuality'] + test_df['DrainageSystems'] + test_df['DeterioratingInfrastructure'] + test_df['RiverManagement']
test_df['EnvRisk'] = test_df['Deforestation'] + test_df['WetlandLoss'] + test_df['Urbanization'] + test_df['Encroachments']
test_df['ClimateRisk'] = test_df['MonsoonIntensity'] + test_df['ClimateChange'] + test_df['Landslides'] + test_df['Siltation']
test_df['RiskRatio'] = test_df['EnvRisk'] / (test_df['InfraScore'] + 1)
test_df['TotalScore'] = test_df[base_features].sum(axis=1)

risk = {0: "Low", 1: "High"}

for i in range(len(test_df)):
    sample = test_df.iloc[i:i+1]
    pred = stacked_model.predict(sample)[0]   # ← stacked_model, not model
    print(risk[pred])


Low
High
High
High
High
